**Archived exploration — not part of the current one-period study.**

# 05 · Uncertainty after 5, 10 and 20 nominal periods
Forecast local covariance with continuing scheduled LSO measurements, using one initial prior and no orbit resets.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "configs/baseline.json").exists() and (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from within the project checkout")
sys.path.insert(0, str(ROOT / "src"))
from halo_tracking.study import load_config
cfg = load_config(ROOT / "configs/baseline.json")
cfg["simulation"]["periods"] = 1.0


## Interpretation
These are zero-innovation covariance forecasts, not Monte Carlo errors. The supplied rounded initial state is propagated continuously without stationkeeping or periodic tiling. It departs the lunar halo region by about 10 nominal periods. Long-span values therefore describe that continued CR3BP trajectory, not a maintained periodic halo. The existing Monte Carlo inconsistency also limits the interpretation of these local covariance predictions.

In [ ]:
from halo_tracking.horizons import forecast_horizons, save_horizon_forecast
periods = [1, 5, 10, 20]
forecast = forecast_horizons(cfg, periods=periods)
save_horizon_forecast(forecast, ROOT / "results/horizons")

## Exact checkpoint values
RSS σ is the square root of the covariance-block trace. The CSV/JSON also contains per-axis position, velocity and acceleration standard deviations. Checkpoints do not add observations off the configured measurement cadence.

In [ ]:
from IPython.display import HTML, display
columns = [("periods", "Periods"), ("days", "Days"), ("measurement_case", "Measurements"), ("position_rss_sigma_km", "Position RSS σ (km)"), ("velocity_rss_sigma_m_s", "Velocity RSS σ (m/s)"), ("acceleration_rss_sigma_m_s2", "Acceleration RSS σ (m/s²)"), ("moon_distance_km", "Moon distance (km)")]
headers = "".join("<th>" + label + "</th>" for _, label in columns)
rows = []
for row in sorted(forecast["horizon_rows"], key=lambda r: (r["periods"], r["measurement_case"])):
    values = [f"{row[key]:.6g}" if isinstance(row[key], (int, float)) else row[key] for key, _ in columns]
    rows.append("<tr>" + "".join("<td>" + value + "</td>" for value in values) + "</tr>")
display(HTML("<table><tr>" + headers + "</tr>" + "".join(rows) + "</table>"))

In [ ]:
from halo_tracking.plotting import horizon_comparison, export
fig = horizon_comparison(forecast)
export(fig, ROOT / "results/horizons/horizon_uncertainty.png")

## Continued trajectory
This figure checks the geometry behind the covariance forecast. Its physical axis scales are equal. Axis-label clearance updates with figure size, camera orientation and export resolution.

In [ ]:
from halo_tracking.plotting import geometry
fig_geometry = geometry(forecast)
export(fig_geometry, ROOT / "results/horizons/continued_trajectory.png")